
## Middleware in LangChain

Middleware is a layer that runs around an agent’s model and tool calls. It allows us to add reusable behavior without rewriting the main agent loop.

Middleware can run before or after a model/tool call and can:

- Retry failed model or tool calls
- Limit model or tool calls
- Summarize long conversations
- Request human approval before sensitive tools run
- Detect or redact PII
- Fall back to another model if the primary model fails
- Add planning and to-do functionality
- Select relevant tools dynamically

Conceptually:

```text
User request
     ↓
Middleware before model call
     ↓
LLM
     ↓
Middleware after model call
     ↓
Tool execution
     ↓
Middleware around tool call
     ↓
Final response
```

Official documentation:

- [LangChain prebuilt middleware](https://docs.langchain.com/oss/python/langchain/middleware/built-in)
- [LangChain middleware reference](https://reference.langchain.com/python/langchain/middleware)
- [LangChain agent middleware overview](https://docs.langchain.com/oss/python/langchain/middleware)
- [LangChain agents documentation](https://docs.langchain.com/oss/python/langchain/agents)

LangChain’s built-in middleware includes tool-error handling, retries, model fallbacks, summarization, human-in-the-loop approval, model-call limits, tool-call limits, PII detection, to-do lists, subagents, and more. 

In [1]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model 
from langchain.agents import create_agent 
load_dotenv()  # Load environment variables from .env file

os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

model = init_chat_model("gpt-4.1") # initialize a chat model with the name "gpt-4"

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.7.3', 'langchain': '1.4.7', 'langchain-openai': '1.7.3'}}, profile={'name': 'GPT-4.1', 'release_date': '2025-04-14', 'last_updated': '2025-04-14', 'open_weights': False, 'max_input_tokens': 1047576, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True, 'mid_conversation_system_messages': True, 'mid_conversation_tools': False}, client=<openai.resources.chat.completions.completions.Completions object at 0x114a13a10>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x114bf8590>

## Summarization Middleware

Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing order context.

Summarization is useful for the following:

- long running conversation that exceed context windows.
- multiturn dialogues with extensive history.
- applications were preserving full conversation context matters.

In [30]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.checkpoint.memory import InMemorySaver #from LangGraph



agent = create_agent(
    model="gpt-4.1",  # The main language model used by the agent.

    # Saves the agent's conversation state and progress.
    # InMemorySaver stores this only in RAM, so it disappears
    # when the Python program stops.
    checkpointer=InMemorySaver(),

    # A list of middleware components that modify or control
    # the agent's behavior.
    middleware=[

        # Automatically summarizes older messages when the
        # conversation becomes too long.
        SummarizationMiddleware(

            # The model used only to create the summaries.
            # This can be a smaller and cheaper model than the
            # main agent model.
            model="gpt-4o-mini",

            # Start summarizing when the conversation reaches
            # 10 messages. -> A message is one complete chat entry, not one token and not necessarily one sentence.
            #
            # "messages" means we are counting messages.
            # "messages" property is built in Summarization middlware so we have to use the property name exactly as "messages"
            # 10 means the summarization threshold.
            trigger=("messages", 10), 
            # After summarization, keep the 5 most recent messages
            # unchanged. Older messages are compressed into a summary.
            keep=("messages", 5),
        )
    ],
)

# We will create a config for a thread id 
config = {"configurable" : {"thread_id":"test-1"}}

messages = [
    "What is 0 + 1",
    "What is prev answer + 2",
    "what is Prev answer + 3",
    "what is Prev answer + 4",
    "What is Prev answer + 5",
    "What is Prev answer + 6",
    "what is Prev answer + 7",
    "what is Prev answer + 8",
]


# An example output of a response object from agent.invoke({}) looks like this: 

# {
#     "messages": [


#         HumanMessage(
#             content="What is the weather in New York?"
#         ),
#         AIMessage(
#             content="",
#             tool_calls=[
#                 {
#                     "name": "get_weather",
#                     "args": {
#                         "city": "New York"
#                     }
#                 }
#             ]
#         ),
#         ToolMessage(
#             content="The weather in New York is sunny."
#         ),
#         AIMessage(
#             content="The weather in New York is sunny."
#         )


#     ]
# }

for q in messages:
    response = agent.invoke({"messages": [HumanMessage(content = q)]}, config)
    print(f"Messages: {response["messages"][-1].content}") #the messages[] list includes, HumanMessage and AIMessage
    print(f"Number of Messages so Far: {len(response['messages'])}") # for each iteration we have 2 messages in the list Human and AI


Messages: 0 + 1 = **1**
Number of Messages so Far: 2
Messages: The previous answer was **1**.

1 + 2 = **3**
Number of Messages so Far: 4
Messages: The previous answer was **3**.

3 + 3 = **6**
Number of Messages so Far: 6
Messages: The previous answer was **6**.

6 + 4 = **10**
Number of Messages so Far: 8
Messages: The previous answer was **10**.

10 + 5 = **15**
Number of Messages so Far: 10
Messages: The previous answer was **15**.

15 + 6 = **21**
Number of Messages so Far: 7
Messages: The previous answer was **21**.

21 + 7 = **28**
Number of Messages so Far: 9
Messages: The previous answer was **28**.

28 + 8 = **36**
Number of Messages so Far: 7


## Doing same With Token Size instead of messages 

In [36]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver #from LangGraph

config = {"configurable" : {"thread_id":"test-1"}}

@tool 
def search_hotels(city: str) -> str:
    """Search Hotels- Return long response to use more tokens"""
    return f"""Hotels in {city}:
    1. Grand Hayat - 5 star, $360/night includes pool, spa, massages, gym, free wifi, breakfast
    1. Holiday Inn - 3 star, $260/night includes pool, spa, free wifi
    1. Fairmont Empress - 4 star, $300/night includes pool, spa, massages, gym, free wifi
    """

agent = create_agent(
    model="gpt-4.1", 
    tools= [search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("tokens", 550), #token isntead of messages, capped at 550
            keep=("tokens", 200), # After summarization, keep the 200 most recent tokens, unchanged. Older messages are compressed into a summary.
        )
    ],
)

def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4 # 4 chars = 1 token

In [37]:
# Run the test 

cities = ["Vancouver", "Banff", "Toronto", "Edmonmton", "San Fransico", "Sacramento", "Las Vegas", "Las Angles"]


for city in cities: 
    response = agent.invoke(
        {
            "messages": [
                HumanMessage(content=f"Find Hotels in this {city}.")
            ]
        },
        config = config
    )

    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response["messages"])} messages")
    print(f"{response["messages"]}")

Vancouver: ~171 tokens, 4 messages
[HumanMessage(content='Find Hotels in this Vancouver.', additional_kwargs={}, response_metadata={}, id='f7a4283b-7486-49da-b0b7-d4f93a760baa'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 55, 'total_tokens': 71, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-2025-04-14', 'system_fingerprint': 'fp_8e73f426bd', 'id': 'chatcmpl-Ege9XrUBQgfgxujmK6waj5ewpEAwo', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a1a8c4-c96a-742d-a4c3-0cb4847d18ec-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Vancouver'}, 'id': 'call_f